# 02. Fast and Slow Pointers (Floyd's Tortoise and Hare)

Two pointers move at **different speeds**: `slow` one step, `fast` two steps. Their meeting or position reveals structure without extra memory.

**Learning order:** MIDDLE → CYCLE → CYCLE START → NTH FROM END → PALINDROME LIST → FLOYD ON ARRAYS → INTERVIEW THINKING

## 1. Real life

Two runners on a track: the faster one will eventually **lap** the slower one **if the track is a loop**. If the track is straight, the faster one just reaches the end first.

When `fast` reaches the end, `slow` is at the **middle** (half the distance).

In [1]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def build(vals):                      # list -> linked list
    dummy = ListNode()
    cur = dummy
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
    return dummy.next

def to_list(head):                    # linked list -> list (for printing)
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out

def middle(head):                     # second middle for even length
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

print(middle(build([1, 2, 3, 4, 5])).val)   # 3
print(middle(build([1, 2, 3, 4])).val)      # 3

3
3


## 2. Detect a cycle

If there is a cycle, `fast` enters it and keeps gaining 1 step on `slow` each turn, so they must meet. No cycle → `fast` hits `None`.

Space O(1). (Using a `set` of visited nodes works but costs O(n) space.)

In [2]:
def has_cycle(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:
            return True
    return False

def make_cycle(vals, pos):            # tail connects to node at index pos (pos = -1: no cycle)
    head = build(vals)
    nodes, cur = [], head
    while cur:
        nodes.append(cur)
        cur = cur.next
    if pos != -1:
        nodes[-1].next = nodes[pos]
    return head

print(has_cycle(make_cycle([3, 2, 0, -4], 1)))   # True
print(has_cycle(make_cycle([1, 2], -1)))         # False

True
False


## 3. Find where the cycle starts

Let `a` = distance from head to cycle start, `b` = distance from cycle start to the meeting point, `L` = cycle length.

- `slow` walked `a + b`. `fast` walked `2(a + b)`, which is also `a + b + kL` (extra full laps).
- So `a + b = kL` → `a = kL - b`.
- From the meeting point, walking `a` more steps lands exactly on the cycle start (it is `L - b` steps ahead, plus full laps).

**Algorithm:** after they meet, put one pointer at `head`. Move both **one step at a time**. They meet at the cycle start.

In [3]:
def cycle_start(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:
            ptr = head
            while ptr is not slow:
                ptr = ptr.next
                slow = slow.next
            return ptr
    return None

h = make_cycle([3, 2, 0, -4], 1)
print(cycle_start(h).val)   # 2

2


## 4. Remove the N-th node from the end

Move `fast` `n` steps ahead. Then move both until `fast` is at the last node. Now `slow` is just **before** the node to delete. Use a dummy node (the head itself may be deleted).

In [4]:
def remove_nth_from_end(head, n):
    dummy = ListNode(0, head)
    slow = fast = dummy
    for _ in range(n):
        fast = fast.next
    while fast.next:
        slow = slow.next
        fast = fast.next
    slow.next = slow.next.next
    return dummy.next

print(to_list(remove_nth_from_end(build([1, 2, 3, 4, 5]), 2)))   # [1, 2, 3, 5]
print(to_list(remove_nth_from_end(build([1]), 1)))               # []

[1, 2, 3, 5]
[]


## 5. Palindrome Linked List

Find the middle, **reverse the second half**, compare with the first half. O(n) time, **O(1) space** (copying to an array is O(n) space).

In [5]:
def reverse(head):
    prev = None
    while head:
        head.next, prev, head = prev, head, head.next
    return prev

def is_palindrome(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    second = reverse(slow)
    first = head
    while second:
        if first.val != second.val:
            return False
        first = first.next
        second = second.next
    return True

print(is_palindrome(build([1, 2, 2, 1])), is_palindrome(build([1, 2])))   # True False

True False


## 6. Floyd on arrays and numbers

Whenever "next" is defined by a function (`i → nums[i]`, or `n → sum of digit squares`), the sequence can loop. Floyd's algorithm finds the loop with O(1) space.

In [6]:
def find_duplicate(nums):             # values in 1..n, n+1 numbers, one repeated
    slow = fast = nums[0]
    while True:
        slow = nums[slow]
        fast = nums[nums[fast]]
        if slow == fast:
            break
    slow = nums[0]
    while slow != fast:
        slow = nums[slow]
        fast = nums[fast]
    return slow

def is_happy(n):
    def nxt(x):
        return sum(int(d) ** 2 for d in str(x))
    slow, fast = n, nxt(n)
    while fast != 1 and slow != fast:
        slow = nxt(slow)
        fast = nxt(nxt(fast))
    return fast == 1

print(find_duplicate([1, 3, 4, 2, 2]), find_duplicate([3, 1, 3, 4, 2]))   # 2 3
print(is_happy(19), is_happy(2))                                          # True False

2 3
True False


## 7. Interview thinking

**Signals:** "middle", "cycle", "start of cycle", "k-th from end", "is palindrome in O(1) space", "repeated value with no extra space".

**Say:** "`fast` moves two steps and `slow` one. If there is a cycle they must meet because `fast` gains one step per iteration. For the start of the cycle I reset one pointer to the head and move both at speed one."

**Common mistakes:** loop condition `while fast.next.next` (crashes), not using a dummy node when deleting the head, comparing values instead of node identity (`is`) for cycles, off-by-one in the n-step gap.

## Quick Revision

- Middle: `while fast and fast.next`.
- Cycle: pointers meet. Start: reset one to head, move both by 1.
- Nth from end: gap of `n`, dummy node.
- Palindrome: middle + reverse second half.
- Works on arrays/numbers when "next" is a function (Find Duplicate, Happy Number).